# Classical Machine Learning

## Random Forest Model Exploration

In [ ]:

"""
Random Forest Model Exploration for Sri Lankan Tourism Arrivals Prediction
===========================================================================
Author: ML Research Team
Date: December 2025
Purpose: Model exploration phase for tourist arrivals forecasting system

This script implements a production-ready Random Forest model with:
- Time-series aware data splitting
- Model-specific feature engineering
- Hyperparameter tuning with cross-validation
- Comprehensive evaluation metrics
- Proper logging and reproducibility
"""

import pandas as pd
import numpy as np
import logging
from datetime import datetime
import warnings
import json
import pickle
from pathlib import Path

# Sklearn imports
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION AND SETUP
# =============================================================================

class Config:
    """Configuration class for model exploration"""

    # Paths
    DATA_PATH = "preprocessed-dataset.csv"
    OUTPUT_DIR = Path("outputs/random_forest_exploration")
    MODEL_DIR = OUTPUT_DIR / "models"
    METRICS_DIR = OUTPUT_DIR / "metrics"
    LOG_DIR = OUTPUT_DIR / "logs"

    # Data splitting ratios
    TRAIN_RATIO = 0.75
    VAL_RATIO = 0.15
    TEST_RATIO = 0.10

    # Model parameters
    RANDOM_STATE = 42
    N_JOBS = -1
    CV_FOLDS = 5

    # Hyperparameter tuning
    N_ITER_SEARCH = 50
    SCORING = 'neg_root_mean_squared_error'

    # Feature columns (as specified by user)
    FEATURE_COLS = [
        'gdp_per_capita', 'brent_crude_price', 'inflation_rate',
        'usd_lkr', 'rub_lkr', 'cny_lkr', 'web_search',
        'image_search', 'temperature', 'humidity', 'precipitation',
        'event_encoded', 'covid_impact_factor', 'crisis_impact_factor',
        'gbp_lkr', 'inr_lkr', 'eur_lkr'
    ]

    TARGET_COL = 'arrivals'
    DATE_COL = 'date'


def setup_logging():
    """Setup logging configuration"""
    Config.LOG_DIR.mkdir(parents=True, exist_ok=True)

    log_file = Config.LOG_DIR / f"rf_exploration_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )

    return logging.getLogger(__name__)


def create_directories():
    """Create necessary output directories"""
    for directory in [Config.OUTPUT_DIR, Config.MODEL_DIR, Config.METRICS_DIR, Config.LOG_DIR]:
        directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================

class DataProcessor:
    """Handle data loading and preprocessing operations"""

    def __init__(self, logger):
        self.logger = logger
        self.scaler = StandardScaler()

    def load_data(self, file_path):
        """Load the preprocessed dataset"""
        self.logger.info(f"Loading data from {file_path}")

        df = pd.read_csv(file_path)
        df[Config.DATE_COL] = pd.to_datetime(df[Config.DATE_COL])
        df = df.sort_values(Config.DATE_COL).reset_index(drop=True)

        self.logger.info(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
        self.logger.info(f"Date range: {df[Config.DATE_COL].min()} to {df[Config.DATE_COL].max()}")

        return df

    def create_rf_specific_features(self, df):
        """
        Create Random Forest specific features
        RF benefits from explicit feature engineering as it doesn't create interactions automatically
        """
        self.logger.info("Creating Random Forest specific features")

        df = df.copy()

        # 1. Temporal Features - Critical for time series
        df['year'] = df[Config.DATE_COL].dt.year
        df['month'] = df[Config.DATE_COL].dt.month
        df['day'] = df[Config.DATE_COL].dt.day
        df['dayofweek'] = df[Config.DATE_COL].dt.dayofweek
        df['dayofyear'] = df[Config.DATE_COL].dt.dayofyear
        df['quarter'] = df[Config.DATE_COL].dt.quarter
        df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
        df['is_month_start'] = df[Config.DATE_COL].dt.is_month_start.astype(int)
        df['is_month_end'] = df[Config.DATE_COL].dt.is_month_end.astype(int)

        # 2. Cyclical encoding for temporal features (captures seasonality better)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
        df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
        df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
        df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)

        # 3. Lagged features (using only features available at prediction time)
        # For production, we'd use confirmed lags only
        lag_features = ['web_search', 'image_search', 'temperature', 'precipitation', 'humidity']
        for feature in lag_features:
            if feature in df.columns:
                df[f'{feature}_lag7'] = df[feature].shift(7)
                df[f'{feature}_lag14'] = df[feature].shift(14)
                df[f'{feature}_lag30'] = df[feature].shift(30)

        # 4. Rolling statistics (capturing trends)
        window_sizes = [7, 14, 30]
        for window in window_sizes:
            for feature in lag_features:
                if feature in df.columns:
                    df[f'{feature}_rolling_mean_{window}'] = df[feature].rolling(window=window, min_periods=1).mean()
                    df[f'{feature}_rolling_std_{window}'] = df[feature].rolling(window=window, min_periods=1).std()

        # 5. Exchange rate interactions (economic relationships)
        df['usd_eur_ratio'] = df['usd_lkr'] / (df['eur_lkr'] + 1e-6)
        df['usd_gbp_ratio'] = df['usd_lkr'] / (df['gbp_lkr'] + 1e-6)
        df['usd_inr_ratio'] = df['usd_lkr'] / (df['inr_lkr'] + 1e-6)

        # 6. Search intensity (Google trends interaction)
        df['search_intensity'] = df['web_search'] * df['image_search']
        df['search_ratio'] = df['web_search'] / (df['image_search'] + 1e-6)

        # 7. Weather comfort index
        df['weather_comfort'] = (df['temperature'] * (100 - df['humidity'])) / 100
        df['is_rainy'] = (df['precipitation'] > 0).astype(int)

        # 8. Economic pressure index
        df['economic_pressure'] = df['inflation_rate'] * df['brent_crude_price'] / df['gdp_per_capita']

        # 9. Crisis interaction
        df['total_crisis_impact'] = df['covid_impact_factor'] + df['crisis_impact_factor']

        # Fill NaN values created by lagging/rolling (use forward fill for initial values)
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df[numeric_cols] = df[numeric_cols].fillna(method='ffill').fillna(method='bfill')

        self.logger.info(f"Feature engineering complete. New shape: {df.shape}")
        self.logger.info(f"Total features created: {df.shape[1] - len(Config.FEATURE_COLS) - 2}")  # -2 for date and target

        return df

    def split_data_timeseries(self, df):
        """
        Split data into train/val/test sets with time-series awareness
        75% train, 15% validation, 10% test
        """
        self.logger.info("Splitting data with time-series awareness")

        n = len(df)
        train_size = int(n * Config.TRAIN_RATIO)
        val_size = int(n * Config.VAL_RATIO)

        train_df = df.iloc[:train_size].copy()
        val_df = df.iloc[train_size:train_size + val_size].copy()
        test_df = df.iloc[train_size + val_size:].copy()

        self.logger.info(f"Train set: {len(train_df)} samples ({train_df[Config.DATE_COL].min()} to {train_df[Config.DATE_COL].max()})")
        self.logger.info(f"Val set: {len(val_df)} samples ({val_df[Config.DATE_COL].min()} to {val_df[Config.DATE_COL].max()})")
        self.logger.info(f"Test set: {len(test_df)} samples ({test_df[Config.DATE_COL].min()} to {test_df[Config.DATE_COL].max()})")

        return train_df, val_df, test_df

    def prepare_features(self, train_df, val_df, test_df):
        """
        Prepare features and target, apply scaling
        """
        self.logger.info("Preparing features and applying scaling")

        # Get all numeric columns except date and target
        feature_cols = [col for col in train_df.columns
                       if col not in [Config.DATE_COL, Config.TARGET_COL]
                       and train_df[col].dtype in [np.float64, np.int64, np.float32, np.int32]]

        self.logger.info(f"Using {len(feature_cols)} features for modeling")

        # Separate features and target
        X_train = train_df[feature_cols].values
        y_train = train_df[Config.TARGET_COL].values

        X_val = val_df[feature_cols].values
        y_val = val_df[Config.TARGET_COL].values

        X_test = test_df[feature_cols].values
        y_test = test_df[Config.TARGET_COL].values

        # Fit scaler on training data only
        self.logger.info("Fitting StandardScaler on training data")
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)
        X_test_scaled = self.scaler.transform(X_test)

        # Save feature names for later use
        self.feature_names = feature_cols

        return (X_train_scaled, y_train), (X_val_scaled, y_val), (X_test_scaled, y_test), feature_cols


# =============================================================================
# MODEL TRAINING AND EVALUATION
# =============================================================================

class RandomForestTrainer:
    """Handle Random Forest model training and evaluation"""

    def __init__(self, logger):
        self.logger = logger
        self.best_model = None
        self.best_params = None

    def define_hyperparameter_space(self):
        """Define hyperparameter search space for Random Forest"""
        param_distributions = {
            'n_estimators': [100, 200, 300, 500, 700],
            'max_depth': [10, 20, 30, 40, 50, None],
            'min_samples_split': [2, 5, 10, 15, 20],
            'min_samples_leaf': [1, 2, 4, 8],
            'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
            'bootstrap': [True, False],
            'max_samples': [0.7, 0.8, 0.9, 1.0],
            'min_impurity_decrease': [0.0, 0.001, 0.01],
        }

        return param_distributions

    def hyperparameter_tuning(self, X_train, y_train):
        """
        Perform hyperparameter tuning using RandomizedSearchCV with TimeSeriesSplit
        """
        self.logger.info("Starting hyperparameter tuning")
        self.logger.info(f"Search iterations: {Config.N_ITER_SEARCH}")

        # Base model
        rf_base = RandomForestRegressor(
            random_state=Config.RANDOM_STATE,
            n_jobs=Config.N_JOBS,
            verbose=0
        )

        # Define search space
        param_distributions = self.define_hyperparameter_space()

        # Time series cross-validation
        tscv = TimeSeriesSplit(n_splits=Config.CV_FOLDS)

        # Randomized search
        random_search = RandomizedSearchCV(
            estimator=rf_base,
            param_distributions=param_distributions,
            n_iter=Config.N_ITER_SEARCH,
            scoring=Config.SCORING,
            cv=tscv,
            random_state=Config.RANDOM_STATE,
            n_jobs=Config.N_JOBS,
            verbose=2,
            return_train_score=True
        )

        self.logger.info("Fitting RandomizedSearchCV...")
        random_search.fit(X_train, y_train)

        self.best_params = random_search.best_params_
        self.best_model = random_search.best_estimator_

        self.logger.info(f"Best parameters found: {self.best_params}")
        self.logger.info(f"Best CV score (RMSE): {-random_search.best_score_:.2f}")

        # Save CV results
        cv_results_df = pd.DataFrame(random_search.cv_results_)
        cv_results_df.to_csv(Config.METRICS_DIR / 'cv_results.csv', index=False)
        self.logger.info(f"CV results saved to {Config.METRICS_DIR / 'cv_results.csv'}")

        return self.best_model, self.best_params

    def train_final_model(self, X_train, y_train, params=None):
        """Train final model with best parameters"""
        self.logger.info("Training final Random Forest model")

        if params is None:
            params = self.best_params

        model = RandomForestRegressor(
            **params,
            random_state=Config.RANDOM_STATE,
            n_jobs=Config.N_JOBS,
            verbose=1
        )

        model.fit(X_train, y_train)

        self.logger.info("Model training complete")
        return model

    def calculate_metrics(self, y_true, y_pred, dataset_name=""):
        """Calculate comprehensive evaluation metrics"""

        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mse = mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred) * 100

        metrics = {
            'RMSE': rmse,
            'MSE': mse,
            'MAE': mae,
            'R2': r2,
            'MAPE': mape
        }

        self.logger.info(f"{dataset_name} Metrics:")
        self.logger.info(f"  RMSE: {rmse:.2f}")
        self.logger.info(f"  MSE: {mse:.2f}")
        self.logger.info(f"  MAE: {mae:.2f}")
        self.logger.info(f"  R²: {r2:.4f}")
        self.logger.info(f"  MAPE: {mape:.2f}%")

        return metrics

    def time_series_cv_evaluation(self, model, X_train, y_train):
        """
        Perform time-series cross-validation evaluation
        """
        self.logger.info(f"Performing time-series CV with {Config.CV_FOLDS} folds")

        tscv = TimeSeriesSplit(n_splits=Config.CV_FOLDS)

        cv_scores = {
            'fold': [],
            'train_r2': [],
            'val_r2': [],
            'train_rmse': [],
            'val_rmse': [],
            'train_mape': [],
            'val_mape': []
        }

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), 1):
            X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
            y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]

            # Train on fold
            model.fit(X_fold_train, y_fold_train)

            # Predictions
            y_train_pred = model.predict(X_fold_train)
            y_val_pred = model.predict(X_fold_val)

            # Calculate metrics
            train_metrics = self.calculate_metrics(y_fold_train, y_train_pred, f"Fold {fold} Train")
            val_metrics = self.calculate_metrics(y_fold_val, y_val_pred, f"Fold {fold} Val")

            cv_scores['fold'].append(fold)
            cv_scores['train_r2'].append(train_metrics['R2'])
            cv_scores['val_r2'].append(val_metrics['R2'])
            cv_scores['train_rmse'].append(train_metrics['RMSE'])
            cv_scores['val_rmse'].append(val_metrics['RMSE'])
            cv_scores['train_mape'].append(train_metrics['MAPE'])
            cv_scores['val_mape'].append(val_metrics['MAPE'])

        cv_df = pd.DataFrame(cv_scores)
        cv_df.to_csv(Config.METRICS_DIR / 'timeseries_cv_scores.csv', index=False)

        self.logger.info("Time-series CV Summary:")
        self.logger.info(f"  Mean Val R²: {np.mean(cv_scores['val_r2']):.4f} (+/- {np.std(cv_scores['val_r2']):.4f})")
        self.logger.info(f"  Mean Val RMSE: {np.mean(cv_scores['val_rmse']):.2f} (+/- {np.std(cv_scores['val_rmse']):.2f})")
        self.logger.info(f"  Mean Val MAPE: {np.mean(cv_scores['val_mape']):.2f}% (+/- {np.std(cv_scores['val_mape']):.2f}%)")

        return cv_df

    def evaluate_model(self, model, X_train, y_train, X_val, y_val, X_test, y_test):
        """Comprehensive model evaluation"""
        self.logger.info("="*60)
        self.logger.info("MODEL EVALUATION")
        self.logger.info("="*60)

        # Predictions
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
        y_test_pred = model.predict(X_test)

        # Calculate metrics for each dataset
        train_metrics = self.calculate_metrics(y_train, y_train_pred, "Training Set")
        val_metrics = self.calculate_metrics(y_val, y_val_pred, "Validation Set")
        test_metrics = self.calculate_metrics(y_test, y_test_pred, "Test Set")

        # Combine all metrics
        all_metrics = {
            'train': train_metrics,
            'validation': val_metrics,
            'test': test_metrics
        }

        return all_metrics, (y_train_pred, y_val_pred, y_test_pred)

    def save_model_artifacts(self, model, scaler, feature_names, metrics, best_params):
        """Save model, scaler, and associated artifacts"""
        self.logger.info("Saving model artifacts")

        # Save model
        model_path = Config.MODEL_DIR / 'random_forest_model.pkl'
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
        self.logger.info(f"Model saved to {model_path}")

        # Save scaler
        scaler_path = Config.MODEL_DIR / 'scaler.pkl'
        with open(scaler_path, 'wb') as f:
            pickle.dump(scaler, f)
        self.logger.info(f"Scaler saved to {scaler_path}")

        # Save feature names
        feature_path = Config.MODEL_DIR / 'feature_names.json'
        with open(feature_path, 'w') as f:
            json.dump({'features': feature_names}, f, indent=2)
        self.logger.info(f"Feature names saved to {feature_path}")

        # Save metrics
        metrics_path = Config.METRICS_DIR / 'model_metrics.json'
        with open(metrics_path, 'w') as f:
            json.dump(metrics, f, indent=2)
        self.logger.info(f"Metrics saved to {metrics_path}")

        # Save best parameters
        params_path = Config.METRICS_DIR / 'best_parameters.json'
        with open(params_path, 'w') as f:
            json.dump(best_params, f, indent=2)
        self.logger.info(f"Best parameters saved to {params_path}")

        # Save feature importance
        if hasattr(model, 'feature_importances_'):
            importance_df = pd.DataFrame({
                'feature': feature_names,
                'importance': model.feature_importances_
            }).sort_values('importance', ascending=False)

            importance_path = Config.METRICS_DIR / 'feature_importance.csv'
            importance_df.to_csv(importance_path, index=False)
            self.logger.info(f"Feature importance saved to {importance_path}")

            # Log top 20 features
            self.logger.info("Top 20 Most Important Features:")
            for idx, row in importance_df.head(20).iterrows():
                self.logger.info(f"  {row['feature']}: {row['importance']:.6f}")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    """Main execution function"""

    # Setup
    create_directories()
    logger = setup_logging()

    logger.info("="*60)
    logger.info("RANDOM FOREST MODEL EXPLORATION - SRI LANKAN TOURISM PREDICTION")
    logger.info("="*60)
    logger.info(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    try:
        # Initialize processors
        data_processor = DataProcessor(logger)
        trainer = RandomForestTrainer(logger)

        # 1. Load Data
        logger.info("\n" + "="*60)
        logger.info("STEP 1: DATA LOADING")
        logger.info("="*60)
        df = data_processor.load_data(Config.DATA_PATH)

        # 2. Feature Engineering
        logger.info("\n" + "="*60)
        logger.info("STEP 2: RANDOM FOREST SPECIFIC FEATURE ENGINEERING")
        logger.info("="*60)
        df_engineered = data_processor.create_rf_specific_features(df)

        # 3. Data Splitting
        logger.info("\n" + "="*60)
        logger.info("STEP 3: TIME-SERIES AWARE DATA SPLITTING")
        logger.info("="*60)
        train_df, val_df, test_df = data_processor.split_data_timeseries(df_engineered)

        # 4. Prepare Features
        logger.info("\n" + "="*60)
        logger.info("STEP 4: FEATURE PREPARATION AND SCALING")
        logger.info("="*60)
        (X_train, y_train), (X_val, y_val), (X_test, y_test), feature_names =             data_processor.prepare_features(train_df, val_df, test_df)

        # 5. Hyperparameter Tuning
        logger.info("\n" + "="*60)
        logger.info("STEP 5: HYPERPARAMETER TUNING")
        logger.info("="*60)
        best_model, best_params = trainer.hyperparameter_tuning(X_train, y_train)

        # 6. Train Final Model
        logger.info("\n" + "="*60)
        logger.info("STEP 6: TRAINING FINAL MODEL")
        logger.info("="*60)
        final_model = trainer.train_final_model(X_train, y_train, best_params)

        # 7. Time-Series Cross-Validation
        logger.info("\n" + "="*60)
        logger.info("STEP 7: TIME-SERIES CROSS-VALIDATION")
        logger.info("="*60)
        cv_results = trainer.time_series_cv_evaluation(final_model, X_train, y_train)

        # 8. Model Evaluation
        logger.info("\n" + "="*60)
        logger.info("STEP 8: COMPREHENSIVE MODEL EVALUATION")
        logger.info("="*60)
        metrics, predictions = trainer.evaluate_model(
            final_model, X_train, y_train, X_val, y_val, X_test, y_test
        )

        # 9. Save Artifacts
        logger.info("\n" + "="*60)
        logger.info("STEP 9: SAVING MODEL ARTIFACTS")
        logger.info("="*60)
        trainer.save_model_artifacts(
            final_model,
            data_processor.scaler,
            feature_names,
            metrics,
            best_params
        )

        # Save predictions for analysis
        y_train_pred, y_val_pred, y_test_pred = predictions

        predictions_df = pd.DataFrame({
            'date': pd.concat([train_df[Config.DATE_COL],
                             val_df[Config.DATE_COL],
                             test_df[Config.DATE_COL]]).values,
            'actual': np.concatenate([y_train, y_val, y_test]),
            'predicted': np.concatenate([y_train_pred, y_val_pred, y_test_pred]),
            'dataset': ['train']*len(y_train) + ['val']*len(y_val) + ['test']*len(y_test)
        })

        predictions_path = Config.METRICS_DIR / 'predictions.csv'
        predictions_df.to_csv(predictions_path, index=False)
        logger.info(f"Predictions saved to {predictions_path}")

        # Final Summary
        logger.info("\n" + "="*60)
        logger.info("EXPLORATION COMPLETE - FINAL SUMMARY")
        logger.info("="*60)
        logger.info(f"Best Model: Random Forest with {best_params.get('n_estimators', 'N/A')} trees")
        logger.info(f"Total Features: {len(feature_names)}")
        logger.info("\nFinal Performance:")
        logger.info(f"  Test R²: {metrics['test']['R2']:.4f}")
        logger.info(f"  Test RMSE: {metrics['test']['RMSE']:.2f}")
        logger.info(f"  Test MAPE: {metrics['test']['MAPE']:.2f}%")
        logger.info(f"\nAll outputs saved to: {Config.OUTPUT_DIR}")
        logger.info(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    except Exception as e:
        logger.error(f"Error during execution: {str(e)}", exc_info=True)
        raise


if __name__ == "__main__":
    main()

Fitting 5 folds for each of 50 candidates, totalling 250 fits


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:    5.2s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   12.8s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:    0.6s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    1.4s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    0.0s finished
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:    1.4s
[Parall

## XGBoost

In [ ]:
import pandas as pd
import numpy as np
import logging
from datetime import datetime
import warnings
import json
import pickle
from pathlib import Path

# XGBoost and Sklearn imports
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION AND SETUP
# =============================================================================

class Config:
    """Configuration class for model exploration"""

    # Paths
    DATA_PATH = "preprocessed-dataset.csv"
    OUTPUT_DIR = Path("outputs/xgboost_exploration")
    MODEL_DIR = OUTPUT_DIR / "models"
    METRICS_DIR = OUTPUT_DIR / "metrics"
    LOG_DIR = OUTPUT_DIR / "logs"

    # Data splitting ratios
    TRAIN_RATIO = 0.75
    VAL_RATIO = 0.15
    TEST_RATIO = 0.10

    # Model parameters
    RANDOM_STATE = 42
    N_JOBS = -1
    CV_FOLDS = 5

    # Hyperparameter tuning
    N_ITER_SEARCH = 50
    SCORING = 'neg_root_mean_squared_error'

    # Early stopping
    EARLY_STOPPING_ROUNDS = 50

    # Feature columns (as specified by user)
    FEATURE_COLS = [
        'gdp_per_capita', 'brent_crude_price', 'inflation_rate',
        'usd_lkr', 'rub_lkr', 'cny_lkr', 'web_search',
        'image_search', 'temperature', 'humidity', 'precipitation',
        'event_encoded', 'covid_impact_factor', 'crisis_impact_factor',
        'gbp_lkr', 'inr_lkr', 'eur_lkr'
    ]

    TARGET_COL = 'arrivals'
    DATE_COL = 'date'


def setup_logging():
    """Setup logging configuration"""
    Config.LOG_DIR.mkdir(parents=True, exist_ok=True)

    log_file = Config.LOG_DIR / f"xgboost_exploration_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )

    return logging.getLogger(__name__)


def create_directories():
    """Create necessary output directories"""
    for directory in [Config.OUTPUT_DIR, Config.MODEL_DIR, Config.METRICS_DIR, Config.LOG_DIR]:
        directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================

class DataProcessor:
    """Handle data loading and preprocessing operations"""

    def __init__(self, logger):
        self.logger = logger
        self.scaler = StandardScaler()

    def load_data(self, file_path):
        """Load the preprocessed dataset"""
        self.logger.info(f"Loading data from {file_path}")

        df = pd.read_csv(file_path)
        df[Config.DATE_COL] = pd.to_datetime(df[Config.DATE_COL])
        df = df.sort_values(Config.DATE_COL).reset_index(drop=True)

        self.logger.info(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
        self.logger.info(f"Date range: {df[Config.DATE_COL].min()} to {df[Config.DATE_COL].max()}")

        return df

    def create_xgboost_specific_features(self, df):
        """
        Create XGBoost specific features
        XGBoost can learn interactions, but explicit features still help
        Focus on temporal patterns and domain-specific transformations
        """
        self.logger.info("Creating XGBoost specific features")

        df = df.copy()

        # 1. Temporal Features - Essential for time series
        df['year'] = df[Config.DATE_COL].dt.year
        df['month'] = df[Config.DATE_COL].dt.month
        df['day'] = df[Config.DATE_COL].dt.day
        df['dayofweek'] = df[Config.DATE_COL].dt.dayofweek
        df['dayofyear'] = df[Config.DATE_COL].dt.dayofyear
        df['quarter'] = df[Config.DATE_COL].dt.quarter
        df['weekofyear'] = df[Config.DATE_COL].dt.isocalendar().week.astype(int)
        df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
        df['is_month_start'] = df[Config.DATE_COL].dt.is_month_start.astype(int)
        df['is_month_end'] = df[Config.DATE_COL].dt.is_month_end.astype(int)
        df['is_quarter_start'] = df[Config.DATE_COL].dt.is_quarter_start.astype(int)
        df['is_quarter_end'] = df[Config.DATE_COL].dt.is_quarter_end.astype(int)

        # 2. Cyclical encoding (helps capture periodicity)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
        df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
        df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
        df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)
        df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
        df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)

        # 3. Lagged features (only features available at prediction time)
        lag_features = ['web_search', 'image_search', 'temperature', 'precipitation', 'humidity']
        lag_periods = [1, 3, 7, 14, 21, 30]

        for feature in lag_features:
            if feature in df.columns:
                for lag in lag_periods:
                    df[f'{feature}_lag{lag}'] = df[feature].shift(lag)

        # 4. Rolling window features (capturing trends and volatility)
        window_sizes = [3, 7, 14, 30, 60, 90]

        for feature in lag_features:
            if feature in df.columns:
                for window in window_sizes:
                    # Mean
                    df[f'{feature}_rolling_mean_{window}'] = df[feature].rolling(
                        window=window, min_periods=1
                    ).mean()
                    # Standard deviation
                    df[f'{feature}_rolling_std_{window}'] = df[feature].rolling(
                        window=window, min_periods=1
                    ).std()
                    # Min and Max
                    df[f'{feature}_rolling_min_{window}'] = df[feature].rolling(
                        window=window, min_periods=1
                    ).min()
                    df[f'{feature}_rolling_max_{window}'] = df[feature].rolling(
                        window=window, min_periods=1
                    ).max()

        # 5. Rate of change features
        for feature in lag_features:
            if feature in df.columns:
                df[f'{feature}_diff1'] = df[feature].diff(1)
                df[f'{feature}_diff7'] = df[feature].diff(7)
                df[f'{feature}_pct_change7'] = df[feature].pct_change(7)
                df[f'{feature}_pct_change30'] = df[feature].pct_change(30)

        # 6. Exchange rate features
        # Ratios
        df['usd_eur_ratio'] = df['usd_lkr'] / (df['eur_lkr'] + 1e-6)
        df['usd_gbp_ratio'] = df['usd_lkr'] / (df['gbp_lkr'] + 1e-6)
        df['usd_inr_ratio'] = df['usd_lkr'] / (df['inr_lkr'] + 1e-6)
        df['eur_gbp_ratio'] = df['eur_lkr'] / (df['gbp_lkr'] + 1e-6)

        # Exchange rate volatility
        for curr in ['usd_lkr', 'eur_lkr', 'gbp_lkr', 'inr_lkr']:
            df[f'{curr}_volatility_7d'] = df[curr].rolling(7).std()
            df[f'{curr}_volatility_30d'] = df[curr].rolling(30).std()

        # 7. Search trend features
        df['search_intensity'] = df['web_search'] * df['image_search']
        df['search_ratio'] = df['web_search'] / (df['image_search'] + 1e-6)
        df['search_avg'] = (df['web_search'] + df['image_search']) / 2
        df['search_diff'] = df['web_search'] - df['image_search']

        # Search momentum
        df['web_search_momentum_7d'] = df['web_search'] - df['web_search'].shift(7)
        df['image_search_momentum_7d'] = df['image_search'] - df['image_search'].shift(7)

        # 8. Weather features
        df['weather_comfort'] = (df['temperature'] * (100 - df['humidity'])) / 100
        df['is_rainy'] = (df['precipitation'] > 0).astype(int)
        df['precipitation_binary'] = (df['precipitation'] > 5).astype(int)  # Heavy rain
        df['temp_humidity_interaction'] = df['temperature'] * df['humidity']

        # 9. Economic indicators
        df['economic_pressure'] = df['inflation_rate'] * df['brent_crude_price'] / (df['gdp_per_capita'] + 1e-6)
        df['oil_price_momentum'] = df['brent_crude_price'].diff(7)
        df['inflation_momentum'] = df['inflation_rate'].diff(30)

        # 10. Crisis and event features
        df['total_crisis_impact'] = df['covid_impact_factor'] + df['crisis_impact_factor']
        df['crisis_interaction'] = df['covid_impact_factor'] * df['crisis_impact_factor']
        df['crisis_weighted'] = df['covid_impact_factor'] * 0.6 + df['crisis_impact_factor'] * 0.4

        # 11. Interaction features (XGBoost can learn these, but explicit helps)
        df['event_search_interaction'] = df['event_encoded'] * df['web_search']
        df['crisis_search_interaction'] = df['total_crisis_impact'] * df['web_search']
        df['weather_search_interaction'] = df['weather_comfort'] * df['web_search']

        # Fill NaN values created by lagging/rolling
        numeric_cols = df.select_dtypes(include=[np.number]).columns

        # Replace infinite values with NaN before filling
        df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

        df[numeric_cols] = df[numeric_cols].fillna(method='ffill').fillna(method='bfill').fillna(0)

        self.logger.info(f"Feature engineering complete. New shape: {df.shape}")
        self.logger.info(f"Total features created: {df.shape[1] - len(Config.FEATURE_COLS) - 2}")

        return df

    def split_data_timeseries(self, df):
        """
        Split data into train/val/test sets with time-series awareness
        75% train, 15% validation, 10% test
        """
        self.logger.info("Splitting data with time-series awareness")

        n = len(df)
        train_size = int(n * Config.TRAIN_RATIO)
        val_size = int(n * Config.VAL_RATIO)

        train_df = df.iloc[:train_size].copy()
        val_df = df.iloc[train_size:train_size + val_size].copy()
        test_df = df.iloc[train_size + val_size:].copy()

        self.logger.info(f"Train set: {len(train_df)} samples ({train_df[Config.DATE_COL].min()} to {train_df[Config.DATE_COL].max()})")
        self.logger.info(f"Val set: {len(val_df)} samples ({val_df[Config.DATE_COL].min()} to {val_df[Config.DATE_COL].max()})")
        self.logger.info(f"Test set: {len(test_df)} samples ({test_df[Config.DATE_COL].min()} to {test_df[Config.DATE_COL].max()})")

        return train_df, val_df, test_df

    def prepare_features(self, train_df, val_df, test_df):
        """
        Prepare features and target, apply scaling
        """
        self.logger.info("Preparing features and applying scaling")

        # Get all numeric columns except date and target
        feature_cols = [col for col in train_df.columns
                       if col not in [Config.DATE_COL, Config.TARGET_COL]
                       and train_df[col].dtype in [np.float64, np.int64, np.float32, np.int32]]

        self.logger.info(f"Using {len(feature_cols)} features for modeling")

        # Separate features and target
        X_train = train_df[feature_cols].values
        y_train = train_df[Config.TARGET_COL].values

        X_val = val_df[feature_cols].values
        y_val = val_df[Config.TARGET_COL].values

        X_test = test_df[feature_cols].values
        y_test = test_df[Config.TARGET_COL].values

        # Fit scaler on training data only
        self.logger.info("Fitting StandardScaler on training data")
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_val_scaled = self.scaler.transform(X_val)
        X_test_scaled = self.scaler.transform(X_test)

        # Save feature names for later use
        self.feature_names = feature_cols

        return (X_train_scaled, y_train), (X_val_scaled, y_val), (X_test_scaled, y_test), feature_cols


# =============================================================================
# MODEL TRAINING AND EVALUATION
# =============================================================================

class XGBoostTrainer:
    """Handle XGBoost model training and evaluation"""

    def __init__(self, logger):
        self.logger = logger
        self.best_model = None
        self.best_params = None

    def define_hyperparameter_space(self):
        """Define hyperparameter search space for XGBoost"""
        param_distributions = {
            # Tree structure
            'n_estimators': [100, 200, 300, 500, 700, 1000],
            'max_depth': [3, 4, 5, 6, 7, 8, 10],
            'min_child_weight': [1, 3, 5, 7, 10],

            # Learning rate
            'learning_rate': [0.001, 0.01, 0.05, 0.1, 0.2, 0.3],

            # Sampling
            'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
            'colsample_bylevel': [0.6, 0.7, 0.8, 0.9, 1.0],

            # Regularization
            'gamma': [0, 0.1, 0.2, 0.5, 1.0],
            'reg_alpha': [0, 0.01, 0.1, 1, 10],
            'reg_lambda': [0.1, 1, 5, 10, 20],
        }

        return param_distributions

    def hyperparameter_tuning(self, X_train, y_train, X_val, y_val):
        """
        Perform hyperparameter tuning using RandomizedSearchCV with TimeSeriesSplit
        """
        self.logger.info("Starting hyperparameter tuning")
        self.logger.info(f"Search iterations: {Config.N_ITER_SEARCH}")

        # Base model
        xgb_base = xgb.XGBRegressor(
            objective='reg:squarederror',
            random_state=Config.RANDOM_STATE,
            n_jobs=Config.N_JOBS,
            tree_method='hist',
            verbosity=0
        )

        # Define search space
        param_distributions = self.define_hyperparameter_space()

        # Time series cross-validation
        tscv = TimeSeriesSplit(n_splits=Config.CV_FOLDS)

        # Randomized search
        random_search = RandomizedSearchCV(
            estimator=xgb_base,
            param_distributions=param_distributions,
            n_iter=Config.N_ITER_SEARCH,
            scoring=Config.SCORING,
            cv=tscv,
            random_state=Config.RANDOM_STATE,
            n_jobs=Config.N_JOBS,
            verbose=2,
            return_train_score=True
        )

        self.logger.info("Fitting RandomizedSearchCV...")
        random_search.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        self.best_params = random_search.best_params_
        self.best_model = random_search.best_estimator_

        self.logger.info(f"Best parameters found: {self.best_params}")
        self.logger.info(f"Best CV score (RMSE): {-random_search.best_score_:.2f}")

        # Save CV results
        cv_results_df = pd.DataFrame(random_search.cv_results_)
        cv_results_df.to_csv(Config.METRICS_DIR / 'cv_results.csv', index=False)
        self.logger.info(f"CV results saved to {Config.METRICS_DIR / 'cv_results.csv'}")

        return self.best_model, self.best_params

    def train_final_model(self, X_train, y_train, X_val, y_val, params=None):
        """Train final model with best parameters and early stopping"""
        self.logger.info("Training final XGBoost model with early stopping")

        if params is None:
            params = self.best_params

        model = xgb.XGBRegressor(
            **params,
            objective='reg:squarederror',
            random_state=Config.RANDOM_STATE,
            n_jobs=Config.N_JOBS,
            tree_method='hist',
            early_stopping_rounds=Config.EARLY_STOPPING_ROUNDS,
            verbosity=1
        )

        model.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
            verbose=True
        )

        self.logger.info(f"Model training complete. Best iteration: {model.best_iteration}")
        self.logger.info(f"Best training score: {model.best_score:.4f}")

        return model

    def calculate_metrics(self, y_true, y_pred, dataset_name=""):
        """Calculate comprehensive evaluation metrics"""

        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mse = mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred) * 100

        metrics = {
            'RMSE': rmse,
            'MSE': mse,
            'MAE': mae,
            'R2': r2,
            'MAPE': mape
        }

        self.logger.info(f"{dataset_name} Metrics:")
        self.logger.info(f"  RMSE: {rmse:.2f}")
        self.logger.info(f"  MSE: {mse:.2f}")
        self.logger.info(f"  MAE: {mae:.2f}")
        self.logger.info(f"  R²: {r2:.4f}")
        self.logger.info(f"  MAPE: {mape:.2f}%")

        return metrics

    def time_series_cv_evaluation(self, X_train, y_train, params):
        """
        Perform time-series cross-validation evaluation
        """
        self.logger.info(f"Performing time-series CV with {Config.CV_FOLDS} folds")

        tscv = TimeSeriesSplit(n_splits=Config.CV_FOLDS)

        cv_scores = {
            'fold': [],
            'train_r2': [],
            'val_r2': [],
            'train_rmse': [],
            'val_rmse': [],
            'train_mape': [],
            'val_mape': []
        }

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), 1):
            X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
            y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]

            # Create and train model for this fold
            model = xgb.XGBRegressor(
                **params,
                objective='reg:squarederror',
                random_state=Config.RANDOM_STATE,
                n_jobs=Config.N_JOBS,
                tree_method='hist',
                verbosity=0
            )

            model.fit(X_fold_train, y_fold_train)

            # Predictions
            y_train_pred = model.predict(X_fold_train)
            y_val_pred = model.predict(X_fold_val)

            # Calculate metrics
            train_metrics = self.calculate_metrics(y_fold_train, y_train_pred, f"Fold {fold} Train")
            val_metrics = self.calculate_metrics(y_fold_val, y_val_pred, f"Fold {fold} Val")

            cv_scores['fold'].append(fold)
            cv_scores['train_r2'].append(train_metrics['R2'])
            cv_scores['val_r2'].append(val_metrics['R2'])
            cv_scores['train_rmse'].append(train_metrics['RMSE'])
            cv_scores['val_rmse'].append(val_metrics['RMSE'])
            cv_scores['train_mape'].append(train_metrics['MAPE'])
            cv_scores['val_mape'].append(val_metrics['MAPE'])

        cv_df = pd.DataFrame(cv_scores)
        cv_df.to_csv(Config.METRICS_DIR / 'timeseries_cv_scores.csv', index=False)

        self.logger.info("Time-series CV Summary:")
        self.logger.info(f"  Mean Val R²: {np.mean(cv_scores['val_r2']):.4f} (+/- {np.std(cv_scores['val_r2']):.4f})")
        self.logger.info(f"  Mean Val RMSE: {np.mean(cv_scores['val_rmse']):.2f} (+/- {np.std(cv_scores['val_rmse']):.2f})")
        self.logger.info(f"  Mean Val MAPE: {np.mean(cv_scores['val_mape']):.2f}% (+/- {np.std(cv_scores['val_mape']):.2f}%)")

        return cv_df

    def evaluate_model(self, model, X_train, y_train, X_val, y_val, X_test, y_test):
        """Comprehensive model evaluation"""
        self.logger.info("="*60)
        self.logger.info("MODEL EVALUATION")
        self.logger.info("="*60)

        # Predictions
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
        y_test_pred = model.predict(X_test)

        # Calculate metrics for each dataset
        train_metrics = self.calculate_metrics(y_train, y_train_pred, "Training Set")
        val_metrics = self.calculate_metrics(y_val, y_val_pred, "Validation Set")
        test_metrics = self.calculate_metrics(y_test, y_test_pred, "Test Set")

        # Combine all metrics
        all_metrics = {
            'train': train_metrics,
            'validation': val_metrics,
            'test': test_metrics
        }

        return all_metrics, (y_train_pred, y_val_pred, y_test_pred)

    def save_model_artifacts(self, model, scaler, feature_names, metrics, best_params):
        """Save model, scaler, and associated artifacts"""
        self.logger.info("Saving model artifacts")

        # Save XGBoost model (native format for better compatibility)
        model_path = Config.MODEL_DIR / 'xgboost_model.json'
        model.save_model(model_path)
        self.logger.info(f"Model saved to {model_path}")

        # Also save as pickle for sklearn compatibility
        model_pkl_path = Config.MODEL_DIR / 'xgboost_model.pkl'
        with open(model_pkl_path, 'wb') as f:
            pickle.dump(model, f)
        self.logger.info(f"Model (pickle) saved to {model_pkl_path}")

        # Save scaler
        scaler_path = Config.MODEL_DIR / 'scaler.pkl'
        with open(scaler_path, 'wb') as f:
            pickle.dump(scaler, f)
        self.logger.info(f"Scaler saved to {scaler_path}")

        # Save feature names
        feature_path = Config.MODEL_DIR / 'feature_names.json'
        with open(feature_path, 'w') as f:
            json.dump({'features': feature_names}, f, indent=2)
        self.logger.info(f"Feature names saved to {feature_path}")

        # Save metrics
        metrics_path = Config.METRICS_DIR / 'model_metrics.json'
        with open(metrics_path, 'w') as f:
            json.dump(metrics, f, indent=2)
        self.logger.info(f"Metrics saved to {metrics_path}")

        # Save best parameters
        params_path = Config.METRICS_DIR / 'best_parameters.json'
        with open(params_path, 'w') as f:
            json.dump(best_params, f, indent=2)
        self.logger.info(f"Best parameters saved to {params_path}")

        # Save feature importance (multiple types for XGBoost)
        if hasattr(model, 'feature_importances_'):
            # Weight importance
            importance_weight_df = pd.DataFrame({
                'feature': feature_names,
                'importance_weight': model.feature_importances_
            }).sort_values('importance_weight', ascending=False)

            # Gain importance
            importance_gain = model.get_booster().get_score(importance_type='gain')
            importance_gain_df = pd.DataFrame([
                {'feature': k, 'importance_gain': v}
                for k, v in importance_gain.items()
            ]).sort_values('importance_gain', ascending=False)

            # Cover importance
            importance_cover = model.get_booster().get_score(importance_type='cover')
            importance_cover_df = pd.DataFrame([
                {'feature': k, 'importance_cover': v}
                for k, v in importance_cover.items()
            ]).sort_values('importance_cover', ascending=False)

            # Merge all importance types
            importance_df = importance_weight_df.copy()
            importance_df = importance_df.merge(
                importance_gain_df, on='feature', how='left'
            ).merge(
                importance_cover_df, on='feature', how='left'
            ).fillna(0)

            importance_path = Config.METRICS_DIR / 'feature_importance.csv'
            importance_df.to_csv(importance_path, index=False)
            self.logger.info(f"Feature importance saved to {importance_path}")

            # Log top 20 features
            self.logger.info("Top 20 Most Important Features (by weight):")
            for idx, row in importance_weight_df.head(20).iterrows():
                self.logger.info(f"  {row['feature']}: {row['importance_weight']:.6f}")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    """Main execution function"""

    # Setup
    create_directories()
    logger = setup_logging()

    logger.info("="*60)
    logger.info("XGBOOST MODEL EXPLORATION - SRI LANKAN TOURISM PREDICTION")
    logger.info("="*60)
    logger.info(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    try:
        # Initialize processors
        data_processor = DataProcessor(logger)
        trainer = XGBoostTrainer(logger)

        # 1. Load Data
        logger.info("\n" + "="*60)
        logger.info("STEP 1: DATA LOADING")
        logger.info("="*60)
        df = data_processor.load_data(Config.DATA_PATH)

        # 2. Feature Engineering
        logger.info("\n" + "="*60)
        logger.info("STEP 2: XGBOOST SPECIFIC FEATURE ENGINEERING")
        logger.info("="*60)
        df_engineered = data_processor.create_xgboost_specific_features(df)

        # 3. Data Splitting
        logger.info("\n" + "="*60)
        logger.info("STEP 3: TIME-SERIES AWARE DATA SPLITTING")
        logger.info("="*60)
        train_df, val_df, test_df = data_processor.split_data_timeseries(df_engineered)

        # 4. Prepare Features
        logger.info("\n" + "="*60)
        logger.info("STEP 4: FEATURE PREPARATION AND SCALING")
        logger.info("="*60)
        (X_train, y_train), (X_val, y_val), (X_test, y_test), feature_names =             data_processor.prepare_features(train_df, val_df, test_df)

        # 5. Hyperparameter Tuning
        logger.info("\n" + "="*60)
        logger.info("STEP 5: HYPERPARAMETER TUNING")
        logger.info("="*60)
        best_model, best_params = trainer.hyperparameter_tuning(X_train, y_train, X_val, y_val)

        # 6. Train Final Model with Early Stopping
        logger.info("\n" + "="*60)
        logger.info("STEP 6: TRAINING FINAL MODEL WITH EARLY STOPPING")
        logger.info("="*60)
        final_model = trainer.train_final_model(X_train, y_train, X_val, y_val, best_params)

        # 7. Time-Series Cross-Validation
        logger.info("\n" + "="*60)
        logger.info("STEP 7: TIME-SERIES CROSS-VALIDATION")
        logger.info("="*60)
        cv_results = trainer.time_series_cv_evaluation(X_train, y_train, best_params)

        # 8. Model Evaluation
        logger.info("\n" + "="*60)
        logger.info("STEP 8: COMPREHENSIVE MODEL EVALUATION")
        logger.info("="*60)
        metrics, predictions = trainer.evaluate_model(
            final_model, X_train, y_train, X_val, y_val, X_test, y_test
        )

        # 9. Save Artifacts
        logger.info("\n" + "="*60)
        logger.info("STEP 9: SAVING MODEL ARTIFACTS")
        logger.info("="*60)
        trainer.save_model_artifacts(
            final_model,
            data_processor.scaler,
            feature_names,
            metrics,
            best_params
        )

        # Save predictions for analysis
        y_train_pred, y_val_pred, y_test_pred = predictions

        predictions_df = pd.DataFrame({
            'date': pd.concat([train_df[Config.DATE_COL],
                             val_df[Config.DATE_COL],
                             test_df[Config.DATE_COL]]).values,
            'actual': np.concatenate([y_train, y_val, y_test]),
            'predicted': np.concatenate([y_train_pred, y_val_pred, y_test_pred]),
            'dataset': ['train']*len(y_train) + ['val']*len(y_val) + ['test']*len(y_test)
        })

        predictions_path = Config.METRICS_DIR / 'predictions.csv'
        predictions_df.to_csv(predictions_path, index=False)
        logger.info(f"Predictions saved to {predictions_path}")

        # Final Summary
        logger.info("\n" + "="*60)
        logger.info("EXPLORATION COMPLETE - FINAL SUMMARY")
        logger.info("="*60)
        logger.info(f"Best Model: XGBoost with {best_params.get('n_estimators', 'N/A')} estimators")
        logger.info(f"Learning Rate: {best_params.get('learning_rate', 'N/A')}")
        logger.info(f"Max Depth: {best_params.get('max_depth', 'N/A')}")
        logger.info(f"Best Iteration (Early Stopping): {final_model.best_iteration}")
        logger.info(f"Total Features: {len(feature_names)}")
        logger.info("\nFinal Performance:")
        logger.info(f"  Test R²: {metrics['test']['R2']:.4f}")
        logger.info(f"  Test RMSE: {metrics['test']['RMSE']:.2f}")
        logger.info(f"  Test MAPE: {metrics['test']['MAPE']:.2f}%")
        logger.info(f"\nAll outputs saved to: {Config.OUTPUT_DIR}")
        logger.info(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    except Exception as e:
        logger.error(f"Error during execution: {str(e)}", exc_info=True)
        raise


if __name__ == "__main__":
    main()

Fitting 5 folds for each of 50 candidates, totalling 250 fits
[0]	validation_0-rmse:2221.04873	validation_1-rmse:1781.11070
[1]	validation_0-rmse:2021.43174	validation_1-rmse:1705.98200
[2]	validation_0-rmse:1839.75030	validation_1-rmse:1656.47002
[3]	validation_0-rmse:1677.96691	validation_1-rmse:1599.05767
[4]	validation_0-rmse:1529.65341	validation_1-rmse:1551.96086
[5]	validation_0-rmse:1395.43963	validation_1-rmse:1506.65795
[6]	validation_0-rmse:1273.80740	validation_1-rmse:1482.58459
[7]	validation_0-rmse:1170.68284	validation_1-rmse:1419.39257
[8]	validation_0-rmse:1074.84464	validation_1-rmse:1392.34818
[9]	validation_0-rmse:983.84126	validation_1-rmse:1374.37585
[10]	validation_0-rmse:906.80434	validation_1-rmse:1359.28743
[11]	validation_0-rmse:835.51056	validation_1-rmse:1358.56827
[12]	validation_0-rmse:771.88157	validation_1-rmse:1366.62795
[13]	validation_0-rmse:716.12716	validation_1-rmse:1365.49381
[14]	validation_0-rmse:662.40353	validation_1-rmse:1370.39961
[15]	vali

## CatBoost

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [ ]:
"""
CatBoost Model Exploration for Sri Lankan Tourism Arrivals Prediction
======================================================================
Author: ML Research Team
Date: December 2025
Purpose: Model exploration phase for tourist arrivals forecasting system

This script implements a production-ready CatBoost model with:
- Time-series aware data splitting
- CatBoost-specific feature engineering
- Hyperparameter tuning with cross-validation
- Comprehensive evaluation metrics
- Proper logging and reproducibility
"""

import pandas as pd
import numpy as np
import logging
from datetime import datetime
import warnings
import json
import pickle
from pathlib import Path

# CatBoost imports
from catboost import CatBoostRegressor, Pool
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION AND SETUP
# =============================================================================

class Config:
    """Configuration class for model exploration"""

    # Paths
    DATA_PATH = "preprocessed-dataset.csv"
    OUTPUT_DIR = Path("outputs/catboost_exploration")
    MODEL_DIR = OUTPUT_DIR / "models"
    METRICS_DIR = OUTPUT_DIR / "metrics"
    LOG_DIR = OUTPUT_DIR / "logs"

    # Data splitting ratios
    TRAIN_RATIO = 0.75
    VAL_RATIO = 0.15
    TEST_RATIO = 0.10

    # Model parameters
    RANDOM_STATE = 42
    N_JOBS = -1
    CV_FOLDS = 5

    # Hyperparameter tuning
    N_ITER_SEARCH = 50

    # Feature columns (as specified by user)
    FEATURE_COLS = [
        'gdp_per_capita', 'brent_crude_price', 'inflation_rate',
        'usd_lkr', 'rub_lkr', 'cny_lkr', 'web_search',
        'image_search', 'temperature', 'humidity', 'precipitation',
        'event_encoded', 'covid_impact_factor', 'crisis_impact_factor',
        'gbp_lkr', 'inr_lkr', 'eur_lkr'
    ]

    TARGET_COL = 'arrivals'
    DATE_COL = 'date'


def setup_logging():
    """Setup logging configuration"""
    Config.LOG_DIR.mkdir(parents=True, exist_ok=True)

    log_file = Config.LOG_DIR / f"catboost_exploration_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )

    return logging.getLogger(__name__)


def create_directories():
    """Create necessary output directories"""
    for directory in [Config.OUTPUT_DIR, Config.MODEL_DIR, Config.METRICS_DIR, Config.LOG_DIR]:
        directory.mkdir(parents=True, exist_ok=True)


# =============================================================================
# DATA LOADING AND PREPROCESSING
# =============================================================================

class DataProcessor:
    """Handle data loading and preprocessing operations"""

    def __init__(self, logger):
        self.logger = logger
        self.scaler = None  # CatBoost often works better without scaling, but we'll keep it optional

    def load_data(self, file_path):
        """Load the preprocessed dataset"""
        self.logger.info(f"Loading data from {file_path}")

        df = pd.read_csv(file_path)
        df[Config.DATE_COL] = pd.to_datetime(df[Config.DATE_COL])
        df = df.sort_values(Config.DATE_COL).reset_index(drop=True)

        self.logger.info(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
        self.logger.info(f"Date range: {df[Config.DATE_COL].min()} to {df[Config.DATE_COL].max()}")

        return df

    def create_catboost_specific_features(self, df):
        """
        Create CatBoost specific features
        CatBoost handles complex interactions internally, but we still add domain knowledge
        """
        self.logger.info("Creating CatBoost specific features")

        df = df.copy()

        # 1. Temporal Features - Essential for time series
        df['year'] = df[Config.DATE_COL].dt.year
        df['month'] = df[Config.DATE_COL].dt.month
        df['day'] = df[Config.DATE_COL].dt.day
        df['dayofweek'] = df[Config.DATE_COL].dt.dayofweek
        df['dayofyear'] = df[Config.DATE_COL].dt.dayofyear
        df['quarter'] = df[Config.DATE_COL].dt.quarter
        df['weekofyear'] = df[Config.DATE_COL].dt.isocalendar().week
        df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
        df['is_month_start'] = df[Config.DATE_COL].dt.is_month_start.astype(int)
        df['is_month_end'] = df[Config.DATE_COL].dt.is_month_end.astype(int)
        df['is_quarter_start'] = df[Config.DATE_COL].dt.is_quarter_start.astype(int)
        df['is_quarter_end'] = df[Config.DATE_COL].dt.is_quarter_end.astype(int)

        # 2. Cyclical encoding (helps gradient boosting models understand circular nature)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
        df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
        df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
        df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)

        # 3. Lagged features (critical for time series forecasting)
        lag_features = ['web_search', 'image_search', 'temperature', 'precipitation',
                       'humidity', 'usd_lkr', 'eur_lkr', 'gbp_lkr']

        for feature in lag_features:
            if feature in df.columns:
                # Multiple lag periods
                df[f'{feature}_lag1'] = df[feature].shift(1)
                df[f'{feature}_lag3'] = df[feature].shift(3)
                df[f'{feature}_lag7'] = df[feature].shift(7)
                df[f'{feature}_lag14'] = df[feature].shift(14)
                df[f'{feature}_lag30'] = df[feature].shift(30)

        # 4. Rolling statistics (trend and volatility indicators)
        window_sizes = [7, 14, 30, 60]
        for window in window_sizes:
            for feature in lag_features:
                if feature in df.columns:
                    # Rolling mean (trend)
                    df[f'{feature}_rolling_mean_{window}'] = df[feature].rolling(
                        window=window, min_periods=1
                    ).mean()

                    # Rolling std (volatility)
                    df[f'{feature}_rolling_std_{window}'] = df[feature].rolling(
                        window=window, min_periods=1
                    ).std()

                    # Rolling min/max (range)
                    df[f'{feature}_rolling_min_{window}'] = df[feature].rolling(
                        window=window, min_periods=1
                    ).min()
                    df[f'{feature}_rolling_max_{window}'] = df[feature].rolling(
                        window=window, min_periods=1
                    ).max()

        # 5. Exponential weighted moving averages (recent trend emphasis)
        for feature in ['web_search', 'image_search', 'usd_lkr']:
            if feature in df.columns:
                df[f'{feature}_ewm_7'] = df[feature].ewm(span=7, adjust=False).mean()
                df[f'{feature}_ewm_30'] = df[feature].ewm(span=30, adjust=False).mean()

        # 6. Rate of change features
        for feature in lag_features:
            if feature in df.columns:
                df[f'{feature}_pct_change_7'] = df[feature].pct_change(periods=7)
                df[f'{feature}_pct_change_30'] = df[feature].pct_change(periods=30)

        # 7. Exchange rate interactions and ratios
        df['usd_eur_ratio'] = df['usd_lkr'] / (df['eur_lkr'] + 1e-6)
        df['usd_gbp_ratio'] = df['usd_lkr'] / (df['gbp_lkr'] + 1e-6)
        df['usd_inr_ratio'] = df['usd_lkr'] / (df['inr_lkr'] + 1e-6)
        df['usd_cny_ratio'] = df['usd_lkr'] / (df['cny_lkr'] + 1e-6)
        df['eur_gbp_ratio'] = df['eur_lkr'] / (df['gbp_lkr'] + 1e-6)

        # Exchange rate volatility
        df['fx_volatility'] = df[['usd_lkr', 'eur_lkr', 'gbp_lkr']].std(axis=1)

        # 8. Search behavior features
        df['search_intensity'] = df['web_search'] * df['image_search']
        df['search_ratio'] = df['web_search'] / (df['image_search'] + 1e-6)
        df['search_sum'] = df['web_search'] + df['image_search']
        df['search_diff'] = df['web_search'] - df['image_search']

        # Search momentum
        df['search_momentum'] = (df['web_search'] - df['web_search'].shift(7)) / (df['web_search'].shift(7) + 1e-6)

        # 9. Weather features
        df['weather_comfort'] = (df['temperature'] * (100 - df['humidity'])) / 100
        df['is_rainy'] = (df['precipitation'] > 0).astype(int)
        df['heavy_rain'] = (df['precipitation'] > 5).astype(int)
        df['temp_humidity_interaction'] = df['temperature'] * df['humidity']

        # Weather extremes
        df['is_hot'] = (df['temperature'] > df['temperature'].quantile(0.75)).astype(int)
        df['is_humid'] = (df['humidity'] > df['humidity'].quantile(0.75)).astype(int)

        # 10. Economic indicators
        df['economic_pressure'] = (df['inflation_rate'] * df['brent_crude_price']) / (df['gdp_per_capita'] + 1e-6)
        df['oil_gdp_ratio'] = df['brent_crude_price'] / (df['gdp_per_capita'] + 1e-6)
        df['inflation_oil_interaction'] = df['inflation_rate'] * df['brent_crude_price']

        # 11. Crisis features
        df['total_crisis_impact'] = df['covid_impact_factor'] + df['crisis_impact_factor']
        df['crisis_squared'] = df['total_crisis_impact'] ** 2
        df['covid_crisis_interaction'] = df['covid_impact_factor'] * df['crisis_impact_factor']

        # 12. Composite indices
        # Tourism attractiveness index
        df['tourism_index'] = (
            df['search_intensity'] / (df['search_intensity'].max() + 1e-6) +
            df['weather_comfort'] / (df['weather_comfort'].max() + 1e-6) +
            (1 - df['total_crisis_impact'] / (df['total_crisis_impact'].max() + 1e-6))
        ) / 3

        # Fill NaN values
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df[numeric_cols] = df[numeric_cols].fillna(method='ffill').fillna(method='bfill').fillna(0)

        # Replace infinite values
        df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], 0)

        self.logger.info(f"Feature engineering complete. New shape: {df.shape}")
        self.logger.info(f"Total features: {df.shape[1] - 2}")  # Excluding date and target

        return df

    def split_data_timeseries(self, df):
        """
        Split data into train/val/test sets with time-series awareness
        75% train, 15% validation, 10% test
        """
        self.logger.info("Splitting data with time-series awareness")

        n = len(df)
        train_size = int(n * Config.TRAIN_RATIO)
        val_size = int(n * Config.VAL_RATIO)

        train_df = df.iloc[:train_size].copy()
        val_df = df.iloc[train_size:train_size + val_size].copy()
        test_df = df.iloc[train_size + val_size:].copy()

        self.logger.info(f"Train set: {len(train_df)} samples ({train_df[Config.DATE_COL].min()} to {train_df[Config.DATE_COL].max()})")
        self.logger.info(f"Val set: {len(val_df)} samples ({val_df[Config.DATE_COL].min()} to {val_df[Config.DATE_COL].max()})")
        self.logger.info(f"Test set: {len(test_df)} samples ({test_df[Config.DATE_COL].min()} to {test_df[Config.DATE_COL].max()})")

        return train_df, val_df, test_df

    def prepare_features(self, train_df, val_df, test_df):
        """
        Prepare features and target
        Note: CatBoost typically works well without scaling, but we keep data unscaled
        """
        self.logger.info("Preparing features (CatBoost - no scaling applied)")

        # Get all numeric columns except date and target
        feature_cols = [col for col in train_df.columns
                       if col not in [Config.DATE_COL, Config.TARGET_COL]
                       and train_df[col].dtype in [np.float64, np.int64, np.float32, np.int32]]

        self.logger.info(f"Using {len(feature_cols)} features for modeling")

        # Separate features and target
        X_train = train_df[feature_cols].values
        y_train = train_df[Config.TARGET_COL].values

        X_val = val_df[feature_cols].values
        y_val = val_df[Config.TARGET_COL].values

        X_test = test_df[feature_cols].values
        y_test = test_df[Config.TARGET_COL].values

        # Save feature names
        self.feature_names = feature_cols

        return (X_train, y_train), (X_val, y_val), (X_test, y_test), feature_cols


# =============================================================================
# MODEL TRAINING AND EVALUATION
# =============================================================================

class CatBoostTrainer:
    """Handle CatBoost model training and evaluation"""

    def __init__(self, logger):
        self.logger = logger
        self.best_model = None
        self.best_params = None
        self.best_iteration = None

    def define_hyperparameter_space(self):
        """Define hyperparameter search space for CatBoost"""
        param_distributions = {
            'iterations': [500, 1000, 1500, 2000, 2500],
            'learning_rate': [0.01, 0.03, 0.05, 0.07, 0.1],
            'depth': [4, 6, 8, 10],
            'l2_leaf_reg': [1, 3, 5, 7, 9],
            'border_count': [32, 64, 128, 254],
            'bagging_temperature': [0, 0.5, 1.0],
            'random_strength': [1, 2, 3],
            'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
            'colsample_bylevel': [0.6, 0.7, 0.8, 0.9, 1.0],
            'min_data_in_leaf': [1, 5, 10, 20],
        }

        return param_distributions

    def random_search_params(self, param_distributions, n_iter):
        """Generate random parameter combinations"""
        param_list = []

        for _ in range(n_iter):
            params = {
                key: np.random.choice(values)
                for key, values in param_distributions.items()
            }
            param_list.append(params)

        return param_list

    def hyperparameter_tuning(self, X_train, y_train, X_val, y_val):
        """
        Perform hyperparameter tuning using manual random search with validation set
        """
        self.logger.info("Starting hyperparameter tuning")
        self.logger.info(f"Search iterations: {Config.N_ITER_SEARCH}")

        param_distributions = self.define_hyperparameter_space()
        param_list = self.random_search_params(param_distributions, Config.N_ITER_SEARCH)

        best_score = float('inf')
        best_params = None
        results = []

        for idx, params in enumerate(param_list, 1):
            self.logger.info(f"\nIteration {idx}/{Config.N_ITER_SEARCH}")
            self.logger.info(f"Testing params: {params}")

            try:
                model = CatBoostRegressor(
                    **params,
                    random_state=Config.RANDOM_STATE,
                    verbose=False,
                    early_stopping_rounds=50,
                    eval_metric='RMSE'
                )

                model.fit(
                    X_train, y_train,
                    eval_set=(X_val, y_val),
                    verbose=False
                )

                # Evaluate on validation set
                y_val_pred = model.predict(X_val)
                val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

                results.append({
                    'iteration': idx,
                    'params': params,
                    'val_rmse': val_rmse,
                    'best_iteration': model.best_iteration_
                })

                self.logger.info(f"Val RMSE: {val_rmse:.2f}, Best iteration: {model.best_iteration_}")

                if val_rmse < best_score:
                    best_score = val_rmse
                    best_params = params.copy()
                    self.best_iteration = model.best_iteration_
                    self.logger.info(f"*** New best score: {best_score:.2f} ***")

            except Exception as e:
                self.logger.warning(f"Error with params {params}: {str(e)}")
                continue

        self.best_params = best_params

        self.logger.info("\n" + "="*60)
        self.logger.info(f"Best parameters found: {self.best_params}")
        self.logger.info(f"Best validation RMSE: {best_score:.2f}")
        self.logger.info(f"Best iteration: {self.best_iteration}")
        self.logger.info("="*60)

        # Save search results
        results_df = pd.DataFrame(results)
        results_df.to_csv(Config.METRICS_DIR / 'hyperparameter_search_results.csv', index=False)
        self.logger.info(f"Search results saved to {Config.METRICS_DIR / 'hyperparameter_search_results.csv'}")

        return self.best_params

    def train_final_model(self, X_train, y_train, X_val, y_val, params=None):
        """Train final model with best parameters"""
        self.logger.info("Training final CatBoost model")

        if params is None:
            params = self.best_params

        model = CatBoostRegressor(
            **params,
            random_state=Config.RANDOM_STATE,
            verbose=True,
            early_stopping_rounds=50,
            eval_metric='RMSE'
        )

        model.fit(
            X_train, y_train,
            eval_set=(X_val, y_val),
            verbose=100
        )

        self.best_model = model
        self.logger.info(f"Model training complete. Best iteration: {model.best_iteration_}")

        return model

    def calculate_metrics(self, y_true, y_pred, dataset_name=""):
        """Calculate comprehensive evaluation metrics"""

        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mse = mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)
        mape = mean_absolute_percentage_error(y_true, y_pred) * 100

        metrics = {
            'RMSE': rmse,
            'MSE': mse,
            'MAE': mae,
            'R2': r2,
            'MAPE': mape
        }

        self.logger.info(f"{dataset_name} Metrics:")
        self.logger.info(f"  RMSE: {rmse:.2f}")
        self.logger.info(f"  MSE: {mse:.2f}")
        self.logger.info(f"  MAE: {mae:.2f}")
        self.logger.info(f"  R²: {r2:.4f}")
        self.logger.info(f"  MAPE: {mape:.2f}%")

        return metrics

    def time_series_cv_evaluation(self, X_train, y_train, params):
        """
        Perform time-series cross-validation evaluation
        """
        self.logger.info(f"Performing time-series CV with {Config.CV_FOLDS} folds")

        tscv = TimeSeriesSplit(n_splits=Config.CV_FOLDS)

        cv_scores = {
            'fold': [],
            'train_r2': [],
            'val_r2': [],
            'train_rmse': [],
            'val_rmse': [],
            'train_mape': [],
            'val_mape': [],
            'best_iteration': []
        }

        for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), 1):
            self.logger.info(f"\nProcessing fold {fold}/{Config.CV_FOLDS}")

            X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
            y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]

            # Train on fold
            model = CatBoostRegressor(
                **params,
                random_state=Config.RANDOM_STATE,
                verbose=False,
                early_stopping_rounds=50,
                eval_metric='RMSE'
            )

            model.fit(
                X_fold_train, y_fold_train,
                eval_set=(X_fold_val, y_fold_val),
                verbose=False
            )

            # Predictions
            y_train_pred = model.predict(X_fold_train)
            y_val_pred = model.predict(X_fold_val)

            # Calculate metrics
            train_metrics = self.calculate_metrics(y_fold_train, y_train_pred, f"Fold {fold} Train")
            val_metrics = self.calculate_metrics(y_fold_val, y_val_pred, f"Fold {fold} Val")

            cv_scores['fold'].append(fold)
            cv_scores['train_r2'].append(train_metrics['R2'])
            cv_scores['val_r2'].append(val_metrics['R2'])
            cv_scores['train_rmse'].append(train_metrics['RMSE'])
            cv_scores['val_rmse'].append(val_metrics['RMSE'])
            cv_scores['train_mape'].append(train_metrics['MAPE'])
            cv_scores['val_mape'].append(val_metrics['MAPE'])
            cv_scores['best_iteration'].append(model.best_iteration_)

        cv_df = pd.DataFrame(cv_scores)
        cv_df.to_csv(Config.METRICS_DIR / 'timeseries_cv_scores.csv', index=False)

        self.logger.info("\nTime-series CV Summary:")
        self.logger.info(f"  Mean Val R²: {np.mean(cv_scores['val_r2']):.4f} (+/- {np.std(cv_scores['val_r2']):.4f})")
        self.logger.info(f"  Mean Val RMSE: {np.mean(cv_scores['val_rmse']):.2f} (+/- {np.std(cv_scores['val_rmse']):.2f})")
        self.logger.info(f"  Mean Val MAPE: {np.mean(cv_scores['val_mape']):.2f}% (+/- {np.std(cv_scores['val_mape']):.2f}%)")
        self.logger.info(f"  Mean Best Iteration: {np.mean(cv_scores['best_iteration']):.0f}")

        return cv_df

    def evaluate_model(self, model, X_train, y_train, X_val, y_val, X_test, y_test):
        """Comprehensive model evaluation"""
        self.logger.info("="*60)
        self.logger.info("MODEL EVALUATION")
        self.logger.info("="*60)

        # Predictions
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
        y_test_pred = model.predict(X_test)

        # Calculate metrics for each dataset
        train_metrics = self.calculate_metrics(y_train, y_train_pred, "Training Set")
        val_metrics = self.calculate_metrics(y_val, y_val_pred, "Validation Set")
        test_metrics = self.calculate_metrics(y_test, y_test_pred, "Test Set")

        # Combine all metrics
        all_metrics = {
            'train': train_metrics,
            'validation': val_metrics,
            'test': test_metrics
        }

        return all_metrics, (y_train_pred, y_val_pred, y_test_pred)

    def save_model_artifacts(self, model, feature_names, metrics, best_params):
        """Save model and associated artifacts"""
        self.logger.info("Saving model artifacts")

        # Save model in CatBoost format
        model_path = Config.MODEL_DIR / 'catboost_model.cbm'
        model.save_model(model_path)
        self.logger.info(f"Model saved to {model_path}")

        # Also save as pickle for compatibility
        model_pkl_path = Config.MODEL_DIR / 'catboost_model.pkl'
        with open(model_pkl_path, 'wb') as f:
            pickle.dump(model, f)
        self.logger.info(f"Model (pickle) saved to {model_pkl_path}")

        # Save feature names
        feature_path = Config.MODEL_DIR / 'feature_names.json'
        with open(feature_path, 'w') as f:
            json.dump({'features': feature_names}, f, indent=2)
        self.logger.info(f"Feature names saved to {feature_path}")

        # Save metrics
        metrics_path = Config.METRICS_DIR / 'model_metrics.json'
        with open(metrics_path, 'w') as f:
            json.dump(metrics, f, indent=2)
        self.logger.info(f"Metrics saved to {metrics_path}")

        # Save best parameters
        params_path = Config.METRICS_DIR / 'best_parameters.json'
        # Convert numpy types to native Python types for JSON serialization
        json_params = {k: int(v) if isinstance(v, (np.integer, np.int64)) else float(v) if isinstance(v, (np.floating, np.float64)) else v
                      for k, v in best_params.items()}
        with open(params_path, 'w') as f:
            json.dump(json_params, f, indent=2)
        self.logger.info(f"Best parameters saved to {params_path}")

        # Save feature importance
        feature_importance = model.get_feature_importance()
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': feature_importance
        }).sort_values('importance', ascending=False)

        importance_path = Config.METRICS_DIR / 'feature_importance.csv'
        importance_df.to_csv(importance_path, index=False)
        self.logger.info(f"Feature importance saved to {importance_path}")

        # Log top 20 features
        self.logger.info("\nTop 20 Most Important Features:")
        for idx, row in importance_df.head(20).iterrows():
            self.logger.info(f"  {row['feature']}: {row['importance']:.6f}")

        # Save SHAP-style feature importance if available
        try:
            shap_values = model.get_feature_importance(type='ShapValues')
            np.save(Config.METRICS_DIR / 'shap_values.npy', shap_values)
            self.logger.info(f"SHAP values saved to {Config.METRICS_DIR / 'shap_values.npy'}")
        except:
            self.logger.info("SHAP values not available for this model")


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def main():
    """Main execution function"""

    # Setup
    create_directories()
    logger = setup_logging()

    logger.info("="*60)
    logger.info("CATBOOST MODEL EXPLORATION - SRI LANKAN TOURISM PREDICTION")
    logger.info("="*60)
    logger.info(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    try:
        # Initialize processors
        data_processor = DataProcessor(logger)
        trainer = CatBoostTrainer(logger)

        # 1. Load Data
        logger.info("\n" + "="*60)
        logger.info("STEP 1: DATA LOADING")
        logger.info("="*60)
        df = data_processor.load_data(Config.DATA_PATH)

        # 2. Feature Engineering
        logger.info("\n" + "="*60)
        logger.info("STEP 2: CATBOOST SPECIFIC FEATURE ENGINEERING")
        logger.info("="*60)
        df_engineered = data_processor.create_catboost_specific_features(df)

        # 3. Data Splitting
        logger.info("\n" + "="*60)
        logger.info("STEP 3: TIME-SERIES AWARE DATA SPLITTING")
        logger.info("="*60)
        train_df, val_df, test_df = data_processor.split_data_timeseries(df_engineered)

        # 4. Prepare Features
        logger.info("\n" + "="*60)
        logger.info("STEP 4: FEATURE PREPARATION")
        logger.info("="*60)
        (X_train, y_train), (X_val, y_val), (X_test, y_test), feature_names =             data_processor.prepare_features(train_df, val_df, test_df)

        # 5. Hyperparameter Tuning
        logger.info("\n" + "="*60)
        logger.info("STEP 5: HYPERPARAMETER TUNING")
        logger.info("="*60)
        best_params = trainer.hyperparameter_tuning(X_train, y_train, X_val, y_val)

        # 6. Train Final Model
        logger.info("\n" + "="*60)
        logger.info("STEP 6: TRAINING FINAL MODEL")
        logger.info("="*60)
        final_model = trainer.train_final_model(X_train, y_train, X_val, y_val, best_params)

        # 7. Time-Series Cross-Validation
        logger.info("\n" + "="*60)
        logger.info("STEP 7: TIME-SERIES CROSS-VALIDATION")
        logger.info("="*60)
        cv_results = trainer.time_series_cv_evaluation(X_train, y_train, best_params)

        # 8. Model Evaluation
        logger.info("\n" + "="*60)
        logger.info("STEP 8: COMPREHENSIVE MODEL EVALUATION")
        logger.info("="*60)
        metrics, predictions = trainer.evaluate_model(
            final_model, X_train, y_train, X_val, y_val, X_test, y_test
        )

        # 9. Save Artifacts
        logger.info("\n" + "="*60)
        logger.info("STEP 9: SAVING MODEL ARTIFACTS")
        logger.info("="*60)
        trainer.save_model_artifacts(
            final_model,
            feature_names,
            metrics,
            best_params
        )

        # Save predictions for analysis
        y_train_pred, y_val_pred, y_test_pred = predictions

        predictions_df = pd.DataFrame({
            'date': pd.concat([train_df[Config.DATE_COL],
                             val_df[Config.DATE_COL],
                             test_df[Config.DATE_COL]]).values,
            'actual': np.concatenate([y_train, y_val, y_test]),
            'predicted': np.concatenate([y_train_pred, y_val_pred, y_test_pred]),
            'dataset': ['train']*len(y_train) + ['val']*len(y_val) + ['test']*len(y_test)
        })

        predictions_path = Config.METRICS_DIR / 'predictions.csv'
        predictions_df.to_csv(predictions_path, index=False)
        logger.info(f"Predictions saved to {predictions_path}")

        # Final Summary
        logger.info("\n" + "="*60)
        logger.info("EXPLORATION COMPLETE - FINAL SUMMARY")
        logger.info("="*60)
        logger.info(f"Model: CatBoost with {best_params.get('iterations', 'N/A')} max iterations")
        logger.info(f"Best iteration: {final_model.best_iteration_}")
        logger.info(f"Total Features: {len(feature_names)}")
        logger.info("\nFinal Performance:")
        logger.info(f"  Test R²: {metrics['test']['R2']:.4f}")
        logger.info(f"  Test RMSE: {metrics['test']['RMSE']:.2f}")
        logger.info(f"  Test MAPE: {metrics['test']['MAPE']:.2f}%")
        logger.info(f"\nAll outputs saved to: {Config.OUTPUT_DIR}")
        logger.info(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    except Exception as e:
        logger.error(f"Error during execution: {str(e)}", exc_info=True)
        raise


if __name__ == "__main__":
    main()

0:	learn: 2241.5073809	test: 1807.7516552	best: 1807.7516552 (0)	total: 104ms	remaining: 1m 44s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1280.463661
bestIteration = 9

Shrink model to first 10 iterations.


## Support Vector Regression (SVR)

In [ ]:
  """
  SVR Model Exploration for Sri Lanka Tourist Arrivals Prediction
  ================================================================
  Author: ML Engineering Team
  Date: December 2025
  Purpose: Model exploration phase for SVR regression with time-series validation
  """

  import pandas as pd
  import numpy as np
  import logging
  from datetime import datetime
  import warnings
  import json
  from pathlib import Path

  # ML Libraries
  from sklearn.svm import SVR
  from sklearn.preprocessing import StandardScaler, RobustScaler
  from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
  from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
  import joblib

  # Suppress warnings for cleaner output
  warnings.filterwarnings('ignore')

  # ============================================================================
  # LOGGING CONFIGURATION
  # ============================================================================

  def setup_logging():
      """Configure logging with both file and console handlers"""
      log_dir = Path('logs')
      log_dir.mkdir(exist_ok=True)

      timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
      log_file = log_dir / f'svr_model_exploration_{timestamp}.log'

      logging.basicConfig(
          level=logging.INFO,
          format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
          handlers=[
              logging.FileHandler(log_file),
              logging.StreamHandler()
          ]
      )
      return logging.getLogger(__name__)

  logger = setup_logging()

  # ============================================================================
  # DATA LOADING AND PREPARATION
  # ============================================================================

  def load_data(file_path):
      """Load preprocessed dataset"""
      logger.info(f"Loading data from {file_path}")
      try:
          df = pd.read_csv(file_path)
          df['date'] = pd.to_datetime(df['date'])
          logger.info(f"Data loaded successfully. Shape: {df.shape}")
          logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")
          return df
      except Exception as e:
          logger.error(f"Error loading data: {str(e)}")
          raise

  # ============================================================================
  # SVR-SPECIFIC PREPROCESSING
  # ============================================================================

  def create_lag_features(df, target_col='arrivals', lags=[7, 14, 30]):
      """
      Create lag features for time-series modeling
      SVR benefits from explicit temporal features
      """
      logger.info("Creating lag features for SVR...")
      df_features = df.copy()

      for lag in lags:
          df_features[f'{target_col}_lag_{lag}'] = df_features[target_col].shift(lag)

      logger.info(f"Created {len(lags)} lag features: {lags}")
      return df_features

  def create_rolling_features(df, target_col='arrivals', windows=[7, 14, 30]):
      """
      Create rolling statistics features
      Captures trend and volatility information
      """
      logger.info("Creating rolling window features...")
      df_features = df.copy()

      for window in windows:
          df_features[f'{target_col}_rolling_mean_{window}'] = (
              df_features[target_col].rolling(window=window, min_periods=1).mean()
          )
          df_features[f'{target_col}_rolling_std_{window}'] = (
              df_features[target_col].rolling(window=window, min_periods=1).std()
          )

      logger.info(f"Created rolling features for windows: {windows}")
      return df_features

  def create_temporal_features(df, date_col='date'):
      """
      Create temporal features from date column
      """
      logger.info("Creating temporal features...")
      df_features = df.copy()

      df_features['day_of_week'] = df_features[date_col].dt.dayofweek
      df_features['day_of_month'] = df_features[date_col].dt.day
      df_features['month'] = df_features[date_col].dt.month
      df_features['quarter'] = df_features[date_col].dt.quarter
      df_features['day_of_year'] = df_features[date_col].dt.dayofyear
      df_features['week_of_year'] = df_features[date_col].dt.isocalendar().week

      # Cyclical encoding for periodic features
      df_features['day_of_week_sin'] = np.sin(2 * np.pi * df_features['day_of_week'] / 7)
      df_features['day_of_week_cos'] = np.cos(2 * np.pi * df_features['day_of_week'] / 7)
      df_features['month_sin'] = np.sin(2 * np.pi * df_features['month'] / 12)
      df_features['month_cos'] = np.cos(2 * np.pi * df_features['month'] / 12)

      logger.info("Temporal features created successfully")
      return df_features

  def prepare_features(df):
      """
      Complete SVR-specific feature engineering pipeline
      """
      logger.info("="*70)
      logger.info("STARTING SVR-SPECIFIC PREPROCESSING")
      logger.info("="*70)

      # Create temporal features
      df = create_temporal_features(df)

      # Create lag features (using shorter lags to minimize data loss)
      df = create_lag_features(df, lags=[7, 14, 30])

      # Create rolling features
      df = create_rolling_features(df, windows=[7, 14, 30])

      # Drop rows with NaN values from lag/rolling features
      initial_rows = len(df)
      df = df.dropna()
      final_rows = len(df)
      logger.info(f"Dropped {initial_rows - final_rows} rows due to NaN in lag/rolling features")
      logger.info(f"Final dataset shape: {df.shape}")

      return df

  # ============================================================================
  # TIME-SERIES AWARE DATA SPLITTING
  # ============================================================================

  def split_data_timeseries(df, train_ratio=0.75, val_ratio=0.15, test_ratio=0.15):
      """
      Split data maintaining temporal order: 75% train, 15% val, 15% test
      """
      logger.info("="*70)
      logger.info("TIME-SERIES AWARE DATA SPLITTING")
      logger.info("="*70)

      n = len(df)
      train_size = int(n * train_ratio)
      val_size = int(n * val_ratio)

      train_df = df.iloc[:train_size].copy()
      val_df = df.iloc[train_size:train_size + val_size].copy()
      test_df = df.iloc[train_size + val_size:].copy()

      logger.info(f"Train set: {len(train_df)} samples ({train_df['date'].min()} to {train_df['date'].max()})")
      logger.info(f"Validation set: {len(val_df)} samples ({val_df['date'].min()} to {val_df['date'].max()})")
      logger.info(f"Test set: {len(test_df)} samples ({test_df['date'].min()} to {test_df['date'].max()})")

      return train_df, val_df, test_df

  def prepare_datasets(train_df, val_df, test_df, target_col='arrivals'):
      """
      Separate features and target, excluding non-feature columns
      """
      # Columns to exclude from features
      exclude_cols = ['date', target_col, 'arrivals_robust_scaled', 'outlier_flag']

      # Get feature columns
      feature_cols = [col for col in train_df.columns if col not in exclude_cols]
      logger.info(f"Number of features: {len(feature_cols)}")
      logger.info(f"Feature columns: {feature_cols}")

      X_train = train_df[feature_cols].values
      y_train = train_df[target_col].values

      X_val = val_df[feature_cols].values
      y_val = val_df[target_col].values

      X_test = test_df[feature_cols].values
      y_test = test_df[target_col].values

      return X_train, y_train, X_val, y_val, X_test, y_test, feature_cols

  # ============================================================================
  # FEATURE SCALING (CRITICAL FOR SVR)
  # ============================================================================

  def scale_features(X_train, X_val, X_test):
      """
      Scale features using StandardScaler (SVR is scale-sensitive)
      Fit only on training data to prevent data leakage
      """
      logger.info("="*70)
      logger.info("FEATURE SCALING")
      logger.info("="*70)

      scaler = StandardScaler()
      X_train_scaled = scaler.fit_transform(X_train)
      X_val_scaled = scaler.transform(X_val)
      X_test_scaled = scaler.transform(X_test)

      logger.info("Features scaled using StandardScaler")
      logger.info(f"Training set - Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.6f}")

      return X_train_scaled, X_val_scaled, X_test_scaled, scaler

  # ============================================================================
  # HYPERPARAMETER TUNING
  # ============================================================================

  def hyperparameter_tuning(X_train, y_train, n_splits=5):
      """
      Perform time-series cross-validation with GridSearchCV
      """
      logger.info("="*70)
      logger.info("HYPERPARAMETER TUNING")
      logger.info("="*70)

      # Define parameter grid for SVR
      param_grid = {
          'kernel': ['rbf', 'linear'],
          'C': [0.1, 1, 10, 100],
          'epsilon': [0.01, 0.1, 0.2],
          'gamma': ['scale', 'auto', 0.001, 0.01]
      }

      logger.info(f"Parameter grid: {param_grid}")
      logger.info(f"Total combinations: {len(param_grid['kernel']) * len(param_grid['C']) * len(param_grid['epsilon']) * len(param_grid['gamma'])}")

      # Time series cross-validator
      tscv = TimeSeriesSplit(n_splits=n_splits)
      logger.info(f"Using TimeSeriesSplit with {n_splits} splits")

      # GridSearchCV
      svr = SVR()
      grid_search = GridSearchCV(
          estimator=svr,
          param_grid=param_grid,
          cv=tscv,
          scoring='neg_mean_squared_error',
          n_jobs=-1,
          verbose=2
      )

      logger.info("Starting GridSearchCV...")
      grid_search.fit(X_train, y_train)

      logger.info("="*70)
      logger.info("HYPERPARAMETER TUNING RESULTS")
      logger.info("="*70)
      logger.info(f"Best parameters: {grid_search.best_params_}")
      logger.info(f"Best CV score (neg_MSE): {grid_search.best_score_:.4f}")
      logger.info(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

      # Log top 5 parameter combinations
      results_df = pd.DataFrame(grid_search.cv_results_)
      results_df = results_df.sort_values('rank_test_score')

      logger.info("\nTop 5 parameter combinations:")
      for idx, row in results_df.head(5).iterrows():
          logger.info(f"  Rank {row['rank_test_score']}: {row['params']} - RMSE: {np.sqrt(-row['mean_test_score']):.4f}")

      return grid_search.best_estimator_, grid_search.best_params_

  # ============================================================================
  # MODEL TRAINING
  # ============================================================================

  def train_model(X_train, y_train, best_params):
      """
      Train final SVR model with best parameters
      """
      logger.info("="*70)
      logger.info("MODEL TRAINING WITH BEST PARAMETERS")
      logger.info("="*70)

      logger.info(f"Training SVR with parameters: {best_params}")

      model = SVR(**best_params)
      model.fit(X_train, y_train)

      logger.info("Model training completed successfully")
      return model

  # ============================================================================
  # MODEL EVALUATION
  # ============================================================================

  def calculate_metrics(y_true, y_pred, dataset_name=""):
      """
      Calculate comprehensive regression metrics
      """
      r2 = r2_score(y_true, y_pred)
      mse = mean_squared_error(y_true, y_pred)
      rmse = np.sqrt(mse)
      mape = mean_absolute_percentage_error(y_true, y_pred) * 100  # Convert to percentage
      mae = np.mean(np.abs(y_true - y_pred))

      metrics = {
          'dataset': dataset_name,
          'r2_score': r2,
          'mse': mse,
          'rmse': rmse,
          'mae': mae,
          'mape': mape
      }

      return metrics

  def evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test):
      """
      Comprehensive model evaluation on all datasets
      """
      logger.info("="*70)
      logger.info("MODEL EVALUATION")
      logger.info("="*70)

      # Predictions
      y_train_pred = model.predict(X_train)
      y_val_pred = model.predict(X_val)
      y_test_pred = model.predict(X_test)

      # Calculate metrics for each dataset
      train_metrics = calculate_metrics(y_train, y_train_pred, "Training")
      val_metrics = calculate_metrics(y_val, y_val_pred, "Validation")
      test_metrics = calculate_metrics(y_test, y_test_pred, "Test")

      # Log results
      for metrics in [train_metrics, val_metrics, test_metrics]:
          logger.info(f"\n{metrics['dataset']} Set Metrics:")
          logger.info(f"  R² Score: {metrics['r2_score']:.6f}")
          logger.info(f"  RMSE: {metrics['rmse']:.4f}")
          logger.info(f"  MSE: {metrics['mse']:.4f}")
          logger.info(f"  MAE: {metrics['mae']:.4f}")
          logger.info(f"  MAPE: {metrics['mape']:.2f}%")

      return {
          'train': train_metrics,
          'validation': val_metrics,
          'test': test_metrics
      }

  # ============================================================================
  # TIME SERIES CROSS-VALIDATION
  # ============================================================================

  def time_series_cross_validation(model, X_train, y_train, n_splits=5):
      """
      Perform time-series cross-validation for robustness check
      """
      logger.info("="*70)
      logger.info("TIME-SERIES CROSS-VALIDATION")
      logger.info("="*70)

      tscv = TimeSeriesSplit(n_splits=n_splits)

      cv_r2_scores = []
      cv_rmse_scores = []
      cv_mape_scores = []

      for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), 1):
          X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
          y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]

          # Train on fold
          fold_model = type(model)(**model.get_params())
          fold_model.fit(X_fold_train, y_fold_train)

          # Predict and evaluate
          y_fold_pred = fold_model.predict(X_fold_val)

          r2 = r2_score(y_fold_val, y_fold_pred)
          rmse = np.sqrt(mean_squared_error(y_fold_val, y_fold_pred))
          mape = mean_absolute_percentage_error(y_fold_val, y_fold_pred) * 100

          cv_r2_scores.append(r2)
          cv_rmse_scores.append(rmse)
          cv_mape_scores.append(mape)

          logger.info(f"Fold {fold}: R²={r2:.4f}, RMSE={rmse:.4f}, MAPE={mape:.2f}%")

      logger.info("\nCross-Validation Summary:")
      logger.info(f"  R² - Mean: {np.mean(cv_r2_scores):.6f}, Std: {np.std(cv_r2_scores):.6f}")
      logger.info(f"  RMSE - Mean: {np.mean(cv_rmse_scores):.4f}, Std: {np.std(cv_rmse_scores):.4f}")
      logger.info(f"  MAPE - Mean: {np.mean(cv_mape_scores):.2f}%, Std: {np.std(cv_mape_scores):.2f}%")

      return {
          'r2': {'mean': np.mean(cv_r2_scores), 'std': np.std(cv_r2_scores), 'scores': cv_r2_scores},
          'rmse': {'mean': np.mean(cv_rmse_scores), 'std': np.std(cv_rmse_scores), 'scores': cv_rmse_scores},
          'mape': {'mean': np.mean(cv_mape_scores), 'std': np.std(cv_mape_scores), 'scores': cv_mape_scores}
      }

  # ============================================================================
  # SAVE ARTIFACTS
  # ============================================================================

  def save_model_artifacts(model, scaler, metrics, cv_results, best_params, feature_cols):
      """
      Save model, scaler, metrics, and metadata
      """
      logger.info("="*70)
      logger.info("SAVING MODEL ARTIFACTS")
      logger.info("="*70)

      # Create output directory
      output_dir = Path('model_artifacts')
      output_dir.mkdir(exist_ok=True)

      timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

      # Save model
      model_path = output_dir / f'svr_model_{timestamp}.pkl'
      joblib.dump(model, model_path)
      logger.info(f"Model saved to {model_path}")

      # Save scaler
      scaler_path = output_dir / f'scaler_{timestamp}.pkl'
      joblib.dump(scaler, scaler_path)
      logger.info(f"Scaler saved to {scaler_path}")

      # Save metrics
      metrics_dict = {
          'train_metrics': metrics['train'],
          'validation_metrics': metrics['validation'],
          'test_metrics': metrics['test'],
          'cv_results': cv_results,
          'best_params': best_params,
          'feature_columns': feature_cols,
          'timestamp': timestamp
      }

      metrics_path = output_dir / f'metrics_{timestamp}.json'
      with open(metrics_path, 'w') as f:
          json.dump(metrics_dict, f, indent=4, default=str)
      logger.info(f"Metrics saved to {metrics_path}")

      # Save predictions for analysis
      predictions_path = output_dir / f'predictions_{timestamp}.csv'
      logger.info(f"Prediction results can be saved to {predictions_path} in production")

      logger.info("All artifacts saved successfully")

  # ============================================================================
  # MAIN EXECUTION PIPELINE
  # ============================================================================

  def main():
      """
      Main execution pipeline for SVR model exploration
      """
      logger.info("="*70)
      logger.info("SVR MODEL EXPLORATION - STARTED")
      logger.info("="*70)
      logger.info(f"Execution started at: {datetime.now()}")

      try:
          # 1. Load data
          df = load_data('preprocessed-dataset.csv')

          # 2. SVR-specific preprocessing
          df_processed = prepare_features(df)

          # 3. Time-series aware data splitting (75/15/15)
          train_df, val_df, test_df = split_data_timeseries(df_processed)

          # 4. Prepare datasets
          X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = prepare_datasets(
              train_df, val_df, test_df
          )

          # 5. Feature scaling (critical for SVR)
          X_train_scaled, X_val_scaled, X_test_scaled, scaler = scale_features(
              X_train, X_val, X_test
          )

          # 6. Hyperparameter tuning
          best_model, best_params = hyperparameter_tuning(X_train_scaled, y_train)

          # 7. Train final model
          final_model = train_model(X_train_scaled, y_train, best_params)

          # 8. Model evaluation
          metrics = evaluate_model(
              final_model,
              X_train_scaled, y_train,
              X_val_scaled, y_val,
              X_test_scaled, y_test
          )

          # 9. Time-series cross-validation
          cv_results = time_series_cross_validation(final_model, X_train_scaled, y_train)

          # 10. Save artifacts
          save_model_artifacts(
              final_model, scaler, metrics, cv_results, best_params, feature_cols
          )

          logger.info("="*70)
          logger.info("SVR MODEL EXPLORATION - COMPLETED SUCCESSFULLY")
          logger.info("="*70)
          logger.info(f"Execution completed at: {datetime.now()}")

          return final_model, scaler, metrics, cv_results

      except Exception as e:
          logger.error(f"ERROR in main execution: {str(e)}", exc_info=True)
          raise

  # ============================================================================
  # ENTRY POINT
  # ============================================================================

  if __name__ == "__main__":
      model, scaler, metrics, cv_results = main()

Fitting 5 folds for each of 96 candidates, totalling 480 fits


# Time Series
